# Optional wind/Ekman sensitivity

Run this notebook only after creating an eddy-day wind-stress cache with geographic `tau_east_pa` and `tau_north_pa`. Wind is treated as a surface-forcing sensitivity, not assumed to be present in the existing BRAN velocity cache.

The prediction is directional: deep-to-surface `TiltDir` should align with recent Ekman transport if wind displaces the shallow structure.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

HERE = Path.cwd()
ANALYSIS_ROOT = HERE.parent
if str(ANALYSIS_ROOT) not in sys.path:
    sys.path.insert(0, str(ANALYSIS_ROOT))
if str(ANALYSIS_ROOT / "beta_effect_background_flow") not in sys.path:
    sys.path.insert(0, str(ANALYSIS_ROOT / "beta_effect_background_flow"))

import seacofs_tilt_tools as tilt
import mechanism_tools as mech

paths = tilt.Paths()
grid = tilt.load_grid(paths.grid, paths.z_r)
df, _ = tilt.load_tilt_tables(paths)
df = mech.require_tilt_measurements(df)
print(f"Rows: {len(df):,}; measured tilts: {df.TiltDis.notna().sum():,}; eddies: {df.Eddy.nunique():,}")


In [ ]:
WIND_CACHE = Path("/srv/scratch/z5297792/SEACOFS_26yr_eddy_dataset/tilt_mechanisms/wind_stress_eddy_day.parquet")
if not WIND_CACHE.exists():
    raise FileNotFoundError(
        f"{WIND_CACHE} does not exist. Build it from geographic surface stress before running this optional analysis."
    )
wind = pd.read_parquet(WIND_CACHE)
data = mech.merge_one_to_one_or_many_to_one(df, wind)
data = tilt.add_pv_gradient_terms(data, grid)
data = mech.add_ekman_transport(data)
display(mech.circular_offset_summary(data, ["tilt_ekman_offset"], group=("Cyc",)))


In [ ]:
# Compare instantaneous alignment by polarity and life stage.
data = tilt.add_time_coordinates(data)
data["life_stage"] = pd.cut(data.norm_time, [0, .25, .75, 1], include_lowest=True,
                            labels=["young", "mature", "decaying"])
display(mech.circular_offset_summary(data, ["tilt_ekman_offset"], group=("Cyc", "life_stage")))


## Required extension

The decisive wind test should use trailing 2, 5, 10 and 20-day stress integrals and compare shallow tilt more strongly than deep tilt. Do not infer wind causation from instantaneous alignment alone, especially because wind and EAC regime both vary seasonally.
